In [81]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

url = os.getenv("DATABASE_URL")

In [82]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT"),
    sslmode="require"
)
print("Connection successful")

Connection successful


# Query 1
Which food items are most depleted and how quickly? This joins inventory with food items and calculates total quantities requested against current stock.

In [83]:
query1 = """
SELECT 
    f.item_name,
    f.category,
    i.stock_quantity AS current_stock,
    SUM(ri.quantity_requested) AS total_requested,
    ROUND(SUM(ri.quantity_requested)::numeric / 
        NULLIF(i.stock_quantity, 0), 2) AS depletion_ratio
FROM food_items f
JOIN inventory i ON f.food_item_id = i.food_item_id
JOIN request_items ri ON f.food_item_id = ri.food_item_id
GROUP BY f.item_name, f.category, i.stock_quantity
ORDER BY depletion_ratio DESC;
"""

df1 = pd.read_sql(query1, conn)
df1

/var/folders/w2/vz5x2_vn38sffvhxw9_c6fqh0000gn/T/ipykernel_15197/2195406658.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)


,item_name,category,current_stock,total_requested,depletion_ratio
0,Bread,fresh,3,5,1.67
1,Baked Beans,tinned,45,50,1.11
2,Baby Formula,baby,6,4,0.67
3,UHT Milk,dairy,8,4,0.50
4,Lentils,dry,12,6,0.50
5,Chopped Tomatoes,tinned,30,14,0.47
6,Tuna,tinned,18,7,0.39
7,Soup,tinned,40,14,0.35
8,Nappies,baby,9,3,0.33
9,Pasta,dry,22,6,0.27


# Query 2
How does volunteer capacity correlate with service output? This joins volunteer shifts with requests by matching on date, showing whether busier volunteer days correspond to higher fulfilment.

In [84]:
query2 = """
SELECT 
    DATE(vs.shift_date) AS activity_date,
    COUNT(DISTINCT vs.volunteer_id) AS volunteers_on_shift,
    SUM(vs.hours_worked) AS total_volunteer_hours,
    COUNT(DISTINCT r.request_id) AS requests_handled,
    COUNT(DISTINCT CASE WHEN r.status = 'fulfilled' 
        THEN r.request_id END) AS fulfilled_requests
FROM volunteer_shifts vs
LEFT JOIN requests r ON DATE(vs.shift_date) = DATE(r.request_date)
GROUP BY DATE(vs.shift_date)
ORDER BY activity_date;
"""

df2 = pd.read_sql(query2, conn)
df2

/var/folders/w2/vz5x2_vn38sffvhxw9_c6fqh0000gn/T/ipykernel_15197/2721051298.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)


,activity_date,volunteers_on_shift,total_volunteer_hours,requests_handled,fulfilled_requests
0,2025-01-10,2,8,0,0
1,2025-01-17,1,3,0,0
2,2025-01-20,1,5,1,1
3,2025-02-03,2,10,0,0
4,2025-02-10,1,4,1,0
5,2025-02-14,1,5,0,0
6,2025-03-05,2,7,0,0
7,2025-03-12,2,10,0,0
8,2025-03-20,1,5,0,0
9,2025-04-01,2,8,0,0


# Query 3
What seasonal demand trends emerge across the year by borough? This joins requests with beneficiaries to show demand volume per borough per month.

In [85]:
query3 = """
SELECT 
    TO_CHAR(r.request_date, 'YYYY-MM') AS month,
    b.borough,
    COUNT(r.request_id) AS total_requests,
    SUM(ri.quantity_requested) AS total_items_requested,
    ROUND(AVG(b.household_size), 2) AS avg_household_size
FROM requests r
JOIN beneficiaries b ON r.beneficiary_id = b.beneficiary_id
JOIN request_items ri ON r.request_id = ri.request_id
GROUP BY TO_CHAR(r.request_date, 'YYYY-MM'), b.borough
ORDER BY month, total_requests DESC;
"""

df3 = pd.read_sql(query3, conn)
df3

/var/folders/w2/vz5x2_vn38sffvhxw9_c6fqh0000gn/T/ipykernel_15197/1239337248.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)


,month,borough,total_requests,total_items_requested,avg_household_size
0,2025-01,Hackney,3,10,4.0
1,2025-01,Tower Hamlets,2,6,2.0
2,2025-02,Lewisham,3,11,3.0
3,2025-02,Southwark,3,12,5.0
4,2025-02,Newham,2,2,1.0
5,2025-03,Hackney,3,13,6.0
6,2025-03,Southwark,3,7,3.0
7,2025-03,Tower Hamlets,2,4,2.0
8,2025-03,Lewisham,2,8,4.0
9,2025-04,Hackney,7,27,4.0
